In [1]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from river import linear_model, optim, preprocessing
# from river.compose import pure_inference_mode

In [2]:
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan
    return 100.0 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [3]:
def build_model():
    model = (
        preprocessing.StandardScaler()
        | linear_model.LinearRegression(
            optimizer=optim.Adam(0.01),
            l2=1e-4,
            intercept_lr=0.01,
        )
    )
    return model

In [4]:
train = pd.read_parquet("../data/gold/train.parquet").copy()
val = pd.read_parquet("../data/gold/val.parquet").copy()

y_col = "Sum of кВт"

for df in [train, val]:
    df["ts"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"],
            hour=df["Hour"] - 1,  # only keep this if Hour is 1..24
        ),
        errors="coerce"
    )

train = train.sort_values("ts").reset_index(drop=True)
val = val.sort_values("ts").reset_index(drop=True)

# Drop rows with invalid timestamps or missing target
train = train.dropna(subset=["ts", y_col]).reset_index(drop=True)
val = val.dropna(subset=["ts", y_col]).reset_index(drop=True)

# Features = everything except target and timestamp
exclude_cols = {y_col, "ts"}
feature_cols = [c for c in train.columns if c not in exclude_cols]

print("Train shape:", train.shape)
print("Val shape:", val.shape)
print("Number of features:", len(feature_cols))
print("First val timestamp:", val["ts"].min())
print("Last val timestamp:", val["ts"].max())

Train shape: (4864324, 633)
Val shape: (304499, 633)
Number of features: 631
First val timestamp: 2025-06-30 00:00:00
Last val timestamp: 2025-07-31 22:00:00


In [5]:
model = build_model()

train_subset = train[feature_cols + [y_col]]

y_true_seen = []
y_pred_seen = []

for step, row in enumerate(
    tqdm(
        train_subset.itertuples(index=False, name=None),
        total=len(train_subset),
        desc="Initial training"
    ),
    start=1
):
    x_vals = row[:-1]
    y = row[-1]
    x = dict(zip(feature_cols, x_vals))

    # run inference every 100k rows on the current row BEFORE training on it
    if step % 100_000 == 0:
        y_pred = model.predict_one(x)
        if y_pred is None:
            y_pred = 0.0

        y_true_seen.append(y)
        y_pred_seen.append(y_pred)

        print(
            f"step={step:,} | "
            f"MAPE={mape(y_true_seen, y_pred_seen):.6f}% | "
            f"SMAPE={smape(y_true_seen, y_pred_seen):.6f}% | "
            f"RMSE={rmse(y_true_seen, y_pred_seen):.6f}"
        )

    model.learn_one(x, y)

Initial training:   0%|          | 0/4864324 [00:00<?, ?it/s]

step=100,000 | MAPE=4.799036% | SMAPE=4.686580% | RMSE=0.880354
step=200,000 | MAPE=6.106381% | SMAPE=6.192851% | RMSE=1.065397
step=300,000 | MAPE=4.293683% | SMAPE=4.352076% | RMSE=0.872341
step=400,000 | MAPE=7.833957% | SMAPE=8.346752% | RMSE=1.301594
step=500,000 | MAPE=7.707299% | SMAPE=8.171321% | RMSE=1.265146
step=600,000 | MAPE=21.246740% | SMAPE=33.505846% | RMSE=2.886648
step=700,000 | MAPE=19.899920% | SMAPE=30.313515% | RMSE=2.876653
step=800,000 | MAPE=18.568470% | SMAPE=27.629271% | RMSE=2.720463
step=900,000 | MAPE=18.920541% | SMAPE=26.737819% | RMSE=2.651129
step=1,000,000 | MAPE=21.008972% | SMAPE=27.383807% | RMSE=3.056803
step=1,100,000 | MAPE=20.697116% | SMAPE=26.363311% | RMSE=2.976303
step=1,200,000 | MAPE=19.188389% | SMAPE=24.379637% | RMSE=2.858250
step=1,300,000 | MAPE=17.877577% | SMAPE=22.671292% | RMSE=2.747532
step=1,400,000 | MAPE=19.177851% | SMAPE=24.196455% | RMSE=2.788517
step=1,500,000 | MAPE=19.584745% | SMAPE=24.512650% | RMSE=3.043468
step=1,6

In [6]:
val = val.copy()
val["date"] = val["ts"].dt.date

daily_results = []
all_preds = []

unique_days = sorted(val["date"].unique())

for day in tqdm(unique_days, desc="Day-ahead validation"):
    day_df = val[val["date"] == day].sort_values("ts")
    day_subset = day_df[["ts", "date"] + feature_cols + [y_col]]

    y_true_day = []
    y_pred_day = []

    # Predict whole day first
    for row in day_subset.itertuples(index=False, name=None):
        ts = row[0]
        date_val = row[1]
        x_vals = row[2:-1]
        y_true = row[-1]

        x = dict(zip(feature_cols, x_vals))
        y_pred = model.predict_one(x)

        if y_pred is None:
            y_pred = 0.0

        y_true_day.append(y_true)
        y_pred_day.append(y_pred)

        all_preds.append({
            "ts": ts,
            "date": date_val,
            "y_true": y_true,
            "y_pred": y_pred,
        })

    # Daily metrics
    day_mape = mape(y_true_day, y_pred_day)
    day_smape = smape(y_true_day, y_pred_day)

    daily_results.append({
        "date": day,
        "n_obs": len(day_df),
        "MAPE": day_mape,
        "SMAPE": day_smape,
        "actual_sum": float(np.sum(y_true_day)),
        "pred_sum": float(np.sum(y_pred_day)),
        "bias_sum": float(np.sum(y_pred_day) - np.sum(y_true_day)),
    })

    # Then learn on that day's true values
    for row in day_subset.itertuples(index=False, name=None):
        x_vals = row[2:-1]
        y = row[-1]
        x = dict(zip(feature_cols, x_vals))
        model.learn_one(x, y)

Day-ahead validation:   0%|          | 0/32 [00:00<?, ?it/s]

In [7]:
daily_metrics = pd.DataFrame(daily_results)
preds_df = pd.DataFrame(all_preds)

# Average of daily metrics
avg_daily_mape = daily_metrics["MAPE"].mean()
avg_daily_smape = daily_metrics["SMAPE"].mean()

# Global metrics across all validation rows
global_mape = mape(preds_df["y_true"], preds_df["y_pred"])
global_smape = smape(preds_df["y_true"], preds_df["y_pred"])

print(f"Average daily MAPE over val:  {avg_daily_mape:.4f}%")
print(f"Average daily SMAPE over val: {avg_daily_smape:.4f}%")
print()
print(f"Global MAPE over all val rows:  {global_mape:.4f}%")
print(f"Global SMAPE over all val rows: {global_smape:.4f}%")

Average daily MAPE over val:  69.9453%
Average daily SMAPE over val: 39.9230%

Global MAPE over all val rows:  69.9587%
Global SMAPE over all val rows: 39.9335%


In [10]:
daily_metrics

,date,n_obs,MAPE,SMAPE,actual_sum,pred_sum,bias_sum
0,2025-06-30,9528,59.526105,33.624965,164838.021808,201054.967231,36216.945423
1,2025-07-01,9528,70.149654,34.173641,165277.739952,184266.875817,18989.135865
2,2025-07-02,9528,72.216976,44.078707,169985.680105,150834.731156,-19150.948949
3,2025-07-03,9528,71.371403,48.860602,174064.934055,141734.655055,-32330.279000
4,2025-07-04,9528,82.341830,53.166565,179212.805551,138988.441903,-40224.363648
5,2025-07-05,9528,83.563735,52.011407,170735.234174,140530.842218,-30204.391956
6,2025-07-06,9528,88.397819,42.909626,168671.545958,191739.118226,23067.572268
7,2025-07-07,9528,89.775405,39.659782,185148.176550,213708.452488,28560.275937
8,2025-07-08,9528,52.340902,37.881391,187766.036727,162411.154610,-25354.882117
9,2025-07-09,9528,61.370111,36.756823,188831.672254,181062.838771,-7768.833483


In [9]:
preds_df.head()

,ts,date,y_true,y_pred
0,2025-06-30,2025-06-30,11.903000,11.344860
1,2025-06-30,2025-06-30,13.570428,20.776931
2,2025-06-30,2025-06-30,12.656830,13.427644
3,2025-06-30,2025-06-30,3.152999,6.891978
4,2025-06-30,2025-06-30,10.385522,15.460999
